# Groundtruth: Multi-Temporal Satellite Image Change Detection

**Colab implementation · PCA + K-means baseline · supervised Random Forest extension**

This notebook takes an earlier and later image of the same area and estimates where visible change occurred. It includes image preparation, change-mask prediction, visualization, evaluation, and a separate train/test demonstration.

> **How to present the methods:** The main PCA + K-means workflow is **unsupervised**. It fits PCA and two clusters anew for each image pair; there is no reusable trained model in that workflow. The Random Forest section is a **supervised extension** that learns from reference masks. It runs here on generated scenes to demonstrate training and held-out testing. Its synthetic scores are not real satellite accuracy.

Run the cells from top to bottom. The demo creates its own small scene pair, so it does not require a download or a GPU. You can optionally upload your own aligned image pair or supply a properly labeled dataset.


## 1. Dataset and pixel values

The project dataset is a collection of eight multi-temporal scene pairs derived from Landsat imagery:

| Scene | Earlier filename/date label | Later filename/date label |
|---|---:|---:|
| Andasol | 09051987 | 09122013 |
| Arcadia Lake | 1986 | 2011 |
| Argentina | 01131994 | 01202014 |
| Ayakkum Lake | 09212003 | 09132012 |
| Cubbie Station | 08311987 | 08232013 |
| Dubai | 11272000 | 11122012 |
| Elephant Butte | 08201991 | 08272011 |
| Goksu River | 07111987 | 07132011 |

These labels come from filenames; they are not a substitute for checking acquisition metadata. In the RGB workflow, each image is decoded into red, green, and blue pixel values from **0–255**. The values are display-image intensities, not calibrated surface reflectance. The described image set has no verified change masks or train/test labels, and the JPEGs do not retain the geospatial metadata needed for area-in-square-metres calculations.

For a reproducible run, this notebook first creates a **synthetic illustration** with a known construction mask. It is deliberately not presented as a Landsat observation. The supervised train/test example also uses generated scenes; replace them with reference-labeled scenes before making real-world performance claims.


## 2. Install dependencies

Colab usually includes most scientific Python packages already. This cell ensures the libraries used below are available.


In [ ]:
%pip -q install numpy pillow scikit-learn matplotlib joblib


In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(suppress=True, precision=4)
print("Libraries ready. Random seed:", SEED)


## 3. Generate a reproducible example pair

The generator makes a textured, map-like landscape. The later view adds a development area, roads, and building blocks. The returned binary reference mask marks the pixels deliberately changed by the generator. That mask lets us illustrate metric calculations; a mask drawn by a person would be needed to evaluate real satellite scenes.


In [ ]:
def make_synthetic_pair(seed=7, size=(400, 300)):
    """Create an illustrative RGB pair and its known binary change mask."""
    width, height = size
    local_rng = np.random.default_rng(seed)

    # Low-frequency color variation gives the background a terrain-like texture.
    coarse = local_rng.normal(0, 1, (max(3, height // 24), max(3, width // 24)))
    coarse_img = Image.fromarray(
        np.uint8(np.clip((coarse - coarse.min()) / (np.ptp(coarse) + 1e-8) * 255, 0, 255))
    ).resize((width, height), Image.Resampling.BICUBIC).filter(ImageFilter.GaussianBlur(5))
    terrain = np.asarray(coarse_img, dtype=np.float32) / 255.0

    base = np.zeros((height, width, 3), dtype=np.uint8)
    base[..., 0] = np.clip(82 + 30 * terrain + local_rng.normal(0, 5, (height, width)), 0, 255)
    base[..., 1] = np.clip(103 + 43 * terrain + local_rng.normal(0, 7, (height, width)), 0, 255)
    base[..., 2] = np.clip(66 + 20 * terrain + local_rng.normal(0, 4, (height, width)), 0, 255)
    before_img = Image.fromarray(base, "RGB")
    draw = ImageDraw.Draw(before_img)

    # Irregular fields and tree patches.
    palette = [(91, 125, 65), (119, 135, 76), (147, 131, 81), (74, 111, 61), (166, 148, 99)]
    cell_w, cell_h = max(28, width // 7), max(26, height // 6)
    for row in range(-1, height // cell_h + 2):
        for col in range(-1, width // cell_w + 2):
            x0 = col * cell_w + int(local_rng.integers(-8, 9))
            y0 = row * cell_h + int(local_rng.integers(-7, 8))
            x1, y1 = x0 + cell_w, y0 + cell_h
            color = palette[int(local_rng.integers(0, len(palette)))]
            draw.polygon([(x0, y0), (x1, y0 + 4), (x1 - 5, y1), (x0 + 4, y1 - 3)], fill=color)
            for offset in range(8, cell_w, 12):
                draw.line((x0 + offset, y0 + 4, x0 + offset - 3, y1 - 3), fill=tuple(int(v * .86) for v in color), width=1)

    # A winding river and clustered vegetation provide stable landmarks.
    river = [(0, int(height * .65)), (int(width * .18), int(height * .59)),
             (int(width * .37), int(height * .69)), (int(width * .58), int(height * .62)),
             (int(width * .78), int(height * .70)), (width, int(height * .64))]
    draw.line(river, fill=(60, 88, 86), width=max(5, width // 48), joint="curve")
    draw.line([(x, y - 2) for x, y in river], fill=(84, 113, 107), width=max(2, width // 120), joint="curve")
    for _ in range(max(80, width // 2)):
        x = int(local_rng.integers(0, width))
        y = int(local_rng.integers(0, height))
        radius = int(local_rng.integers(2, 6))
        green = int(local_rng.integers(52, 96))
        draw.ellipse((x - radius, y - radius, x + radius, y + radius), fill=(35, green, 43))

    before = np.asarray(before_img, dtype=np.uint8)
    # Small acquisition noise is present outside the known construction area.
    after = np.clip(before.astype(np.int16) + local_rng.normal(0, 1.0, before.shape), 0, 255).astype(np.uint8)
    after_img = Image.fromarray(after, "RGB")
    after_draw = ImageDraw.Draw(after_img)
    reference_img = Image.new("L", (width, height), 0)
    mask_draw = ImageDraw.Draw(reference_img)

    # Move the site by scene seed so the classifier cannot rely on one fixed location.
    site_w = int(width * .27)
    site_h = int(height * .32)
    site_x = int(local_rng.uniform(.53, .69) * width)
    site_y = int(local_rng.uniform(.48, .60) * height)
    site_x = min(site_x, width - site_w - 2)
    site_y = min(site_y, height - site_h - 2)
    site = [(site_x + 5, site_y + 4), (site_x + site_w - 7, site_y),
            (site_x + site_w, site_y + site_h - 9), (site_x + 12, site_y + site_h)]
    after_draw.polygon(site, fill=(178, 147, 100))
    mask_draw.polygon(site, fill=255)

    road = max(2, width // 100)
    after_draw.line((site_x + 6, site_y + site_h // 2, site_x + site_w - 5, site_y + site_h // 2), fill=(74, 77, 70), width=road + 1)
    after_draw.line((site_x + site_w // 2, site_y + 4, site_x + site_w // 2, site_y + site_h - 3), fill=(74, 77, 70), width=road + 1)
    # Building blocks are drawn on top of the road grid.
    house_step = max(7, width // 28)
    for yy in range(site_y + 8, site_y + site_h - 3, house_step):
        for xx in range(site_x + 8, site_x + site_w - 3, house_step):
            if abs(xx - (site_x + site_w // 2)) < road + 3 or abs(yy - (site_y + site_h // 2)) < road + 3:
                continue
            shade = int(local_rng.integers(185, 226))
            after_draw.rectangle((xx, yy, xx + 3, yy + 3), fill=(shade, shade - 8, shade - 18))

    return before, np.asarray(after_img, dtype=np.uint8), np.asarray(reference_img) >= 128


before, after, reference_mask = make_synthetic_pair(seed=7, size=(400, 300))
print(f"Earlier: {before.shape[1]} × {before.shape[0]} RGB")
print(f"Later:   {after.shape[1]} × {after.shape[0]} RGB")
print(f"Reference changed pixels: {int(reference_mask.sum()):,}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, image, title in zip(axes, [before, after, reference_mask], ["Earlier image", "Later image", "Synthetic reference mask"]):
    ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()


## 4. Unsupervised prediction: RGB difference → PCA → K-means

For corresponding pixels, the implementation computes absolute RGB differences. PCA projects those three values into two components; K-means forms two groups. The cluster with the higher average RGB difference is called the candidate-change cluster. A 3 × 3 majority filter removes isolated pixels.

This is an **unsupervised per-pair fit**. It detects visual difference, not a semantic class such as “new house,” and it cannot distinguish construction from shadows, season, haze, or misregistration by itself.


In [ ]:
def majority_filter(mask):
    """3 × 3 majority smoothing, using only valid image neighbors at borders."""
    height, width = mask.shape
    padded = np.pad(mask.astype(np.uint8), 1, mode="constant", constant_values=0)
    valid = np.pad(np.ones((height, width), dtype=np.uint8), 1, mode="constant", constant_values=0)
    changed_neighbors = np.zeros((height, width), dtype=np.uint8)
    valid_neighbors = np.zeros((height, width), dtype=np.uint8)
    for dy in range(3):
        for dx in range(3):
            changed_neighbors += padded[dy:dy + height, dx:dx + width]
            valid_neighbors += valid[dy:dy + height, dx:dx + width]
    return changed_neighbors * 2 > valid_neighbors


def detect_change_pca_kmeans(before_rgb, after_rgb, sensitivity=50):
    """Return a binary mask and summary from one aligned RGB image pair."""
    if before_rgb.shape != after_rgb.shape or before_rgb.ndim != 3 or before_rgb.shape[2] != 3:
        raise ValueError("The earlier and later images must have identical H×W×3 RGB shapes.")
    if not 0 <= sensitivity <= 100:
        raise ValueError("Sensitivity must be an integer from 0 to 100.")

    before_f = before_rgb.astype(np.float32) / 255.0
    after_f = after_rgb.astype(np.float32) / 255.0
    delta = np.abs(after_f - before_f)
    height, width, _ = delta.shape
    magnitude = np.sqrt(np.mean(delta ** 2, axis=2)).reshape(-1)
    difference_vectors = delta.reshape(-1, 3)

    pca = PCA(n_components=2, svd_solver="full")
    features = pca.fit_transform(difference_vectors).astype(np.float32, copy=False)
    initial = features[[int(np.argmin(magnitude)), int(np.argmax(magnitude))]]
    kmeans = KMeans(n_clusters=2, init=initial, n_init=1, max_iter=100,
                    tol=1e-6, random_state=SEED)
    labels = kmeans.fit_predict(features)
    means = np.array([magnitude[labels == group].mean() if np.any(labels == group) else 0.0 for group in range(2)])
    changed_cluster = int(np.argmax(means))
    stable_cluster = 1 - changed_cluster
    centers = kmeans.cluster_centers_
    center_gap = float(np.linalg.norm(centers[changed_cluster] - centers[stable_cluster]))

    # Sensitivity shifts the decision boundary; it is not an accuracy estimate.
    margin = ((sensitivity - 50) / 100.0) * center_gap * 0.5
    to_stable = np.linalg.norm(features - centers[stable_cluster], axis=1)
    to_changed = np.linalg.norm(features - centers[changed_cluster], axis=1)
    predicted = (to_stable - to_changed > -margin) & (magnitude > 0.012)
    if float(magnitude.max(initial=0.0)) < 0.003 or center_gap < 1e-8:
        predicted[:] = False

    mask = majority_filter(predicted.reshape(height, width))
    stats = {
        "width": width,
        "height": height,
        "changed_pixels": int(mask.sum()),
        "changed_percent": round(100.0 * float(mask.mean()), 3),
        "mean_rgb_difference": round(float(magnitude.mean()), 6),
        "sensitivity": sensitivity,
        "pca_explained_variance_ratio": np.round(pca.explained_variance_ratio_, 4).tolist(),
        "cluster_mean_differences": np.round(means, 6).tolist(),
        "changed_cluster": changed_cluster,
    }
    return mask, magnitude.reshape(height, width), stats


predicted_mask, difference_magnitude, prediction_summary = detect_change_pca_kmeans(
    before, after, sensitivity=50
)
print(json.dumps(prediction_summary, indent=2))


In [ ]:
def make_overlay(base_rgb, mask):
    overlay = (base_rgb.astype(np.float32) * 0.72).astype(np.uint8)
    red = np.zeros_like(overlay)
    red[..., 0] = 255
    overlay[mask] = (0.45 * overlay[mask] + 0.55 * red[mask]).astype(np.uint8)
    return overlay


fig, axes = plt.subplots(1, 5, figsize=(19, 4))
views = [before, after, difference_magnitude, predicted_mask, make_overlay(after, predicted_mask)]
titles = ["Earlier image", "Later image", "RGB difference magnitude", "Predicted change mask", "Prediction over later image"]
for ax, image, title in zip(axes, views, titles):
    ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()


## 5. Evaluate a prediction against a reference mask

The equations below compare each predicted pixel with the corresponding reference pixel. White/nonzero in a mask means **changed**; black means **stable**.

- **TP:** predicted changed and reference changed
- **FP:** predicted changed but reference stable
- **FN:** predicted stable but reference changed
- **TN:** predicted stable and reference stable

Accuracy can look high when stable pixels dominate, so also inspect precision, recall, F1, and Intersection over Union (IoU). The values in this cell are a **synthetic demonstration only** because the reference mask was generated together with the sample images.


In [ ]:
def evaluate_masks(truth, prediction):
    truth = np.asarray(truth, dtype=bool).reshape(-1)
    prediction = np.asarray(prediction, dtype=bool).reshape(-1)
    if truth.shape != prediction.shape:
        raise ValueError("Reference and predicted masks must have the same number of pixels.")
    tn, fp, fn, tp = (int(x) for x in confusion_matrix(truth, prediction, labels=[False, True]).ravel())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    iou = tp / (tp + fp + fn) if tp + fp + fn else 0.0
    total = tp + tn + fp + fn
    return {
        "accuracy": (tp + tn) / total if total else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "true_negative": tn,
    }


sample_metrics = evaluate_masks(reference_mask, predicted_mask)
print("Synthetic sample comparison — not real satellite accuracy")
for name in ("accuracy", "precision", "recall", "f1", "iou"):
    print(f"{name:>10}: {sample_metrics[name]:.4f} ({sample_metrics[name] * 100:.2f}%)")
print("Confusion counts:", {key: sample_metrics[key] for key in ("true_positive", "false_positive", "false_negative", "true_negative")})


## 6. Supervised extension: train and test a Random Forest

Unlike the per-pair PCA + K-means method, this extension learns a reusable pixel classifier from labeled examples. Each pixel gets ten features: 3 RGB values from the earlier date, 3 from the later date, 3 absolute channel differences, and 1 RGB change magnitude.

Scenes are split **by scene** before fitting, so pixels from a test scene do not leak into training. This notebook uses generated scene/mask pairs to make the training code runnable without annotations. Its score measures recovery of generated masks, not performance on Landsat or another real sensor.


In [ ]:
FEATURE_NAMES = [
    "before_r", "before_g", "before_b",
    "after_r", "after_g", "after_b",
    "absolute_delta_r", "absolute_delta_g", "absolute_delta_b",
    "rgb_change_magnitude",
]


def make_pixel_features(before_rgb, after_rgb):
    if before_rgb.shape != after_rgb.shape or before_rgb.ndim != 3 or before_rgb.shape[2] != 3:
        raise ValueError("Expected same-sized RGB image arrays.")
    before_f = before_rgb.astype(np.float32) / 255.0
    after_f = after_rgb.astype(np.float32) / 255.0
    delta = np.abs(after_f - before_f)
    magnitude = np.sqrt(np.mean(delta * delta, axis=2, keepdims=True))
    return np.concatenate((before_f, after_f, delta, magnitude), axis=2).reshape(-1, len(FEATURE_NAMES))


def sample_balanced_pixels(scene, rng, max_per_class=1200):
    features = make_pixel_features(scene["before"], scene["after"])
    labels = scene["mask"].reshape(-1).astype(np.uint8)
    chosen = []
    for class_value in (0, 1):
        indices = np.flatnonzero(labels == class_value)
        if len(indices) == 0:
            raise ValueError(f"Scene {scene['id']} is missing class {class_value} in its reference mask.")
        if len(indices) > max_per_class:
            indices = rng.choice(indices, max_per_class, replace=False)
        chosen.append(indices)
    indices = np.concatenate(chosen)
    rng.shuffle(indices)
    return features[indices], labels[indices]


def fit_random_forest(training_scenes, seed=42, max_per_class=1200):
    sample_rng = np.random.default_rng(seed)
    pairs = [sample_balanced_pixels(scene, sample_rng, max_per_class) for scene in training_scenes]
    x_train = np.concatenate([pair[0] for pair in pairs], axis=0)
    y_train = np.concatenate([pair[1] for pair in pairs], axis=0)
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=16,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=seed,
    )
    model.fit(x_train, y_train)
    return model, x_train.shape, np.bincount(y_train, minlength=2)


def predict_random_forest(model, before_rgb, after_rgb, threshold=0.5):
    if not 0 <= threshold <= 1:
        raise ValueError("Probability threshold must be from 0 to 1.")
    features = make_pixel_features(before_rgb, after_rgb)
    positive_column = int(np.flatnonzero(model.classes_ == 1)[0])
    probabilities = model.predict_proba(features)[:, positive_column]
    return probabilities.reshape(before_rgb.shape[:2]) >= threshold


def evaluate_scenes(model, scenes, threshold=0.5):
    all_truth, all_prediction, per_scene = [], [], {}
    for scene in scenes:
        prediction = predict_random_forest(model, scene["before"], scene["after"], threshold)
        per_scene[scene["id"]] = evaluate_masks(scene["mask"], prediction)
        all_truth.append(scene["mask"].reshape(-1))
        all_prediction.append(prediction.reshape(-1))
    overall = evaluate_masks(np.concatenate(all_truth), np.concatenate(all_prediction))
    return overall, per_scene


In [ ]:
# Create 20 independently seeded scenes with known masks for a runnable demonstration.
synthetic_scenes = []
for scene_number in range(20):
    scene_before, scene_after, scene_mask = make_synthetic_pair(
        seed=100 + scene_number, size=(128, 128)
    )
    synthetic_scenes.append({
        "id": f"synthetic_scene_{scene_number + 1:02d}",
        "before": scene_before,
        "after": scene_after,
        "mask": scene_mask,
    })

train_scenes, test_scenes = train_test_split(
    synthetic_scenes, test_size=0.25, random_state=SEED, shuffle=True
)
print(f"Training scenes: {len(train_scenes)}")
print(f"Held-out test scenes: {len(test_scenes)}")
print("Train scene IDs:", [scene["id"] for scene in train_scenes])
print("Test scene IDs: ", [scene["id"] for scene in test_scenes])

random_forest, training_shape, training_class_counts = fit_random_forest(
    train_scenes, seed=SEED, max_per_class=1200
)
print(f"\nRandom Forest trained on {training_shape[0]:,} sampled pixels × {training_shape[1]} features")
print("Sampled training labels [stable, changed]:", training_class_counts.tolist())


In [ ]:
test_metrics, metrics_by_scene = evaluate_scenes(random_forest, test_scenes, threshold=0.5)
print("Held-out synthetic test metrics — generated scenes only")
for name in ("accuracy", "precision", "recall", "f1", "iou"):
    print(f"{name:>10}: {test_metrics[name]:.4f} ({test_metrics[name] * 100:.2f}%)")
print("Confusion counts:", {key: test_metrics[key] for key in ("true_positive", "false_positive", "false_negative", "true_negative")})
print("\nPer-scene metrics:")
for scene_id, metrics in metrics_by_scene.items():
    print(f"{scene_id}: F1={metrics['f1']:.3f}, IoU={metrics['iou']:.3f}")


In [ ]:
example_test_scene = test_scenes[0]
example_rf_mask = predict_random_forest(
    random_forest, example_test_scene["before"], example_test_scene["after"]
)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
views = [example_test_scene["before"], example_test_scene["after"], example_test_scene["mask"], example_rf_mask]
titles = ["Test scene: earlier", "Test scene: later", "Reference mask", "Random Forest prediction"]
for ax, image, title in zip(axes, views, titles):
    ax.imshow(image, cmap="gray" if image.ndim == 2 else None, vmin=0, vmax=1 if image.ndim == 2 else None)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()


## 7. Use the supervised workflow with real labeled scenes

The described Landsat-derived image set has no verified reference masks, so it cannot train or fairly test this Random Forest yet. To use this section with real labels, prepare a folder like this and keep each complete scene in only one split:

```text
dataset/
  train/
    scene_001/before.jpg
    scene_001/after.jpg
    scene_001/mask.png
    scene_002/...
  test/
    scene_101/before.jpg
    scene_101/after.jpg
    scene_101/mask.png
```

All three files in a scene must cover the same aligned pixel grid. The binary reference mask must use black for stable and white for changed. Do not use an image from `test/` to tune the threshold or model settings. In Colab, upload/extract your labeled ZIP or mount Drive, set `REAL_DATA_ROOT`, then run the cell below.


In [ ]:
def read_labeled_scenes(split_dir, max_side=800):
    """Load scene subfolders containing before.jpg, after.jpg, and mask.png."""
    split_dir = Path(split_dir)
    if not split_dir.is_dir():
        raise NotADirectoryError(f"Could not find dataset split folder: {split_dir}")
    scene_folders = sorted(path for path in split_dir.iterdir() if path.is_dir())
    if not scene_folders:
        raise ValueError(f"No scene folders found in {split_dir}")
    loaded = []
    for folder in scene_folders:
        before_path, after_path, mask_path = folder / "before.jpg", folder / "after.jpg", folder / "mask.png"
        missing = [path.name for path in (before_path, after_path, mask_path) if not path.is_file()]
        if missing:
            raise FileNotFoundError(f"{folder.name}: missing {', '.join(missing)}")
        before_img = Image.open(before_path).convert("RGB")
        after_img = Image.open(after_path).convert("RGB")
        mask_img = Image.open(mask_path).convert("L")
        if before_img.size != after_img.size or before_img.size != mask_img.size:
            raise ValueError(f"{folder.name}: before, after, and mask sizes must match.")
        width, height = before_img.size
        scale = min(1.0, max_side / max(width, height))
        if scale < 1.0:
            new_size = (max(1, round(width * scale)), max(1, round(height * scale)))
            before_img = before_img.resize(new_size, Image.Resampling.LANCZOS)
            after_img = after_img.resize(new_size, Image.Resampling.LANCZOS)
            mask_img = mask_img.resize(new_size, Image.Resampling.NEAREST)
        raw_mask = np.asarray(mask_img, dtype=np.uint8)
        truth = raw_mask > 0 if raw_mask.max(initial=0) <= 1 else raw_mask >= 128
        loaded.append({
            "id": folder.name,
            "before": np.asarray(before_img, dtype=np.uint8),
            "after": np.asarray(after_img, dtype=np.uint8),
            "mask": truth,
        })
    return loaded


# Example after uploading a labeled dataset ZIP to Colab and extracting it:
# from google.colab import drive
# drive.mount('/content/drive')
# REAL_DATA_ROOT = Path('/content/drive/MyDrive/change_detection_dataset')
# real_train_scenes = read_labeled_scenes(REAL_DATA_ROOT / 'train')
# real_test_scenes = read_labeled_scenes(REAL_DATA_ROOT / 'test')
# real_model, real_training_shape, real_class_counts = fit_random_forest(real_train_scenes)
# real_test_metrics, real_per_scene = evaluate_scenes(real_model, real_test_scenes)
# print(json.dumps(real_test_metrics, indent=2))
# print("These metrics are meaningful only if test scenes were kept out of training and tuning.")


## 8. Predict on a custom image pair

The main unsupervised workflow can analyze an uploaded pair without labels. Both images must show the same area, be aligned, and have the same dimensions. The optional Colab cell below asks for two image files and applies PCA + K-means. Add a mask as a third file only if you have a reference mask for metric calculation.


In [ ]:
# Set this to True and run the cell to select custom files in Colab.
RUN_CUSTOM_UPLOAD = False

if RUN_CUSTOM_UPLOAD:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("This upload widget is available when the notebook is running in Google Colab.") from exc

    uploaded_files = files.upload()  # Select before image, after image, and optionally a binary mask.
    available = {name.lower(): name for name in uploaded_files}
    before_name = next((name for key, name in available.items() if "before" in key), None)
    after_name = next((name for key, name in available.items() if "after" in key), None)
    if before_name is None or after_name is None:
        raise ValueError("Name the two uploaded files with 'before' and 'after' in their filenames.")

    custom_before = np.asarray(Image.open(before_name).convert("RGB"), dtype=np.uint8)
    custom_after = np.asarray(Image.open(after_name).convert("RGB"), dtype=np.uint8)
    if custom_before.shape != custom_after.shape:
        raise ValueError("Resize/crop and align the two images to identical dimensions first.")
    custom_mask, custom_difference, custom_summary = detect_change_pca_kmeans(
        custom_before, custom_after, sensitivity=50
    )
    print(json.dumps(custom_summary, indent=2))
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    for ax, image, title in zip(
        axes,
        [custom_before, custom_after, custom_mask, make_overlay(custom_after, custom_mask)],
        ["Earlier", "Later", "Predicted mask", "Overlay"],
    ):
        ax.imshow(image, cmap="gray" if image.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()

    mask_name = next((name for key, name in available.items() if "mask" in key), None)
    if mask_name:
        custom_truth = np.asarray(Image.open(mask_name).convert("L")) >= 128
        if custom_truth.shape != custom_mask.shape:
            raise ValueError("The uploaded reference mask must have the same dimensions as the image pair.")
        print("Metrics against supplied mask:", json.dumps(evaluate_masks(custom_truth, custom_mask), indent=2))
    else:
        print("No reference mask uploaded: accuracy, precision, recall, F1, and IoU are not measured.")
else:
    print("Custom upload is off. Set RUN_CUSTOM_UPLOAD = True to analyze your own aligned image pair.")


## 9. Save the trained model and final interpretation

The object below is the supervised Random Forest trained on **synthetic generated scenes**. Save it only as a demonstration artifact. A model trained on the real project dataset requires verified masks and should be saved with the dataset description, feature definition, split IDs, threshold, and evaluation results.


In [ ]:
DEMO_MODEL_PATH = Path("/content/groundtruth_random_forest_synthetic_demo.joblib")
joblib.dump({
    "model": random_forest,
    "features": FEATURE_NAMES,
    "probability_threshold": 0.5,
    "training_scenes": [scene["id"] for scene in train_scenes],
    "test_scenes": [scene["id"] for scene in test_scenes],
    "note": "Synthetic-data demo only; not a real Landsat model.",
}, DEMO_MODEL_PATH)
print("Saved synthetic demonstration model:", DEMO_MODEL_PATH)


### Conclusions

1. The unsupervised baseline produces a candidate change map for one image pair; it does **not** train once and then generalize as a learned model.
2. The supervised Random Forest extension has an actual training phase and a held-out, scene-level test phase. This notebook uses generated masks only so the whole workflow runs immediately.
3. The synthetic test scores are not a claim about satellite accuracy. Real evaluation needs human-verified masks and scenes untouched during training and tuning.
4. RGB differences are sensitive to alignment, lighting, seasonal vegetation, haze, and sensor processing. The output indicates where pixels look different; it does not explain why or identify objects.
5. The workflow uses image pixels. It does not preserve GeoTIFF coordinates or calculate physical area without geospatial metadata.

**For a project presentation:** show the two dates, RGB difference, PCA + K-means mask, overlay, confusion counts, and the scene-level train/test example. State clearly which outputs are synthetic demonstrations and which, if any, come from real labeled data.
